# Offline run analysis

Set `TRAIN_RUN_ID` and `TEST_RUN_ID` to choose where the training and test raw `.npz` files come from. The test file can come from the same run, a different run, a run folder used only for testing, or an explicit raw `.npz` path via `TEST_RAW_NPZ_OVERRIDE`. This notebook applies its own preprocessing and audio-cue labeling settings, trains offline model variants, tests each on the selected test recording, and saves all notebook-03 outputs under `runs_offline/<OFFLINE_RUN_ID>/`. Models use time-domain EEG windows after this notebook's preprocessing step.


In [ ]:
# imports 
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'training.py').exists():
    ROOT = Path(r'D:/BME/BCI/online_bci/online_eeg')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from preprocessing import (
    AudioLabelConfig,
    PreprocessConfig,
    labeled_preprocess_summary,
    preprocess_recording,
)
from training import (
    TrainingConfig,
    offline_train_test_sweep,
    rank_sweep_by_causal_delay,
    rank_sweep_summary,
    select_lowest_causal_delay_variant,
)
from testing import load_test_variant_artifacts
from plots import (
    plot_labeled_recording,
    plot_offline_variant_trace_and_xcov,
    plot_predictions_overlay,
    plot_xcov_curve,
)

print('Pipeline root:', ROOT)


## Raw files, preprocessing, and sweep settings


In [ ]:
TRAIN_RUN_ID = 'run_002'
TEST_RUN_ID = TRAIN_RUN_ID  # Change this to use another run as the test set.
RUN_ID = TRAIN_RUN_ID  # Kept for titles and older cells.
RUNS_ROOT = ROOT / 'runs'
TRAIN_RUN_DIR = RUNS_ROOT / TRAIN_RUN_ID
TEST_RUN_DIR = RUNS_ROOT / TEST_RUN_ID

OFFLINE_RUN_ID = 'offline_002'  # Change this to keep a separate set of 03 outputs.
OFFLINE_ROOT = ROOT / 'runs_offline'
OFFLINE_RUN_DIR = OFFLINE_ROOT / OFFLINE_RUN_ID

TRAIN_RAW_SUBDIR = 'raw_training'
TRAIN_RAW_PATTERN = f'{TRAIN_RUN_ID}_train.npz'
TRAIN_RAW_NPZ_OVERRIDE = None  # Set to a full Path/string to use any raw training .npz.

TEST_RAW_SUBDIR = 'realtime_trials'  # Use 'raw_training' to test on a training-style recording from another run.
TEST_RAW_PATTERN = '*_raw.npz'  # Matches raw realtime trials such as realtime_trial_01_raw.npz.
TEST_RAW_NPZ_OVERRIDE = "runs\run_003\raw_training\run_003_train.npz"  # Set to a full Path/string to use any raw test .npz.

def resolve_one_raw_npz(run_id, run_dir, subdir, pattern, override, label):
    if override is not None:
        path = Path(override)
        if not path.exists():
            raise FileNotFoundError(f'{label} raw override not found: {path}')
        return path
    if not run_dir.exists():
        raise FileNotFoundError(f'{label} run folder not found for {run_id}: {run_dir}')
    raw_dir = run_dir / subdir
    matches = sorted(path for path in raw_dir.glob(pattern) if path.is_file())
    if not matches:
        raise FileNotFoundError(
            f'No {label} raw files matched {raw_dir / pattern}. '
            'Adjust the run ID, subdir, pattern, or override path.'
        )
    if len(matches) > 1:
        raise ValueError(
            f'Multiple {label} raw files matched {raw_dir / pattern}: '
            + ', '.join(str(path) for path in matches)
            + '. Narrow the pattern or set the override path.'
        )
    return matches[0]

TRAIN_RAW_NPZ = resolve_one_raw_npz(
    TRAIN_RUN_ID,
    TRAIN_RUN_DIR,
    TRAIN_RAW_SUBDIR,
    TRAIN_RAW_PATTERN,
    TRAIN_RAW_NPZ_OVERRIDE,
    'training',
)
TEST_RAW_NPZ = resolve_one_raw_npz(
    TEST_RUN_ID,
    TEST_RUN_DIR,
    TEST_RAW_SUBDIR,
    TEST_RAW_PATTERN,
    TEST_RAW_NPZ_OVERRIDE,
    'test',
)

EEG_CHANNELS = (1, 2, 3, 4)
EOG_CHANNELS = (5,)
EEG_CHANNEL_NAMES = ('O1', 'Oz', 'O2', 'POz')
AUDIO_CHANNEL = 16

APPLY_SOFTWARE_FILTERS = True  # BIOPAC hardware already bandpasses the EEG at 1-35 Hz.
DEMEAN_CHANNELS = True
SOFTWARE_BANDPASS_HZ = (8.0, 35.0)  # Set to (8.0, 35.0) to filter out blinks only if APPLY_SOFTWARE_FILTERS=True.
SOFTWARE_NOTCH_HZ = (60.0,)  # Set to (60.0,) only if APPLY_SOFTWARE_FILTERS=True.
PREPROCESS_TAG = 'software_filters_on' if APPLY_SOFTWARE_FILTERS else 'hardware_filter_only'
if DEMEAN_CHANNELS:
    PREPROCESS_TAG += '_demeaned'

OFFLINE_LABELED_DIR = OFFLINE_RUN_DIR / 'labeled'
TRAIN_LABELED_NPZ = OFFLINE_LABELED_DIR / f'{TRAIN_RUN_ID}__{TRAIN_RAW_NPZ.stem}_labeled.npz'
TEST_LABELED_NPZ = OFFLINE_LABELED_DIR / f'{TEST_RUN_ID}__{TEST_RAW_NPZ.stem}_labeled.npz'
SWEEP_DIR = OFFLINE_RUN_DIR / 'sweeps'

FEATURE_MODES = ('filtered_signal',)

WINDOW_SECS = (1.0, 2.0,)
# STRIDE_SECS = (0.05, 0.1, 0.2)
STRIDE_SECS = (0.02, 0.05, 0.1,)
LABEL_MODES = ('endpoint', 'majority')  # Include one or both: 'endpoint', 'majority'.

PRE = PreprocessConfig(
    eeg_channels=EEG_CHANNELS,
    eog_channels=EOG_CHANNELS,
    audio_channel=AUDIO_CHANNEL,
    apply_software_filters=APPLY_SOFTWARE_FILTERS,
    bandpass_low_hz=None if SOFTWARE_BANDPASS_HZ is None else SOFTWARE_BANDPASS_HZ[0],
    bandpass_high_hz=None if SOFTWARE_BANDPASS_HZ is None else SOFTWARE_BANDPASS_HZ[1],
    notch_hz=SOFTWARE_NOTCH_HZ,
    notch_quality_factor=30.0,
    filter_order=4,
    demean_channels=DEMEAN_CHANNELS,
)

LABELS = AudioLabelConfig(
    class_names=('Eyes Open', 'Eyes Closed'),
    baseline_label=0,
    active_label=1,
    cue_label_sequence=None,
    alternate_binary_labels=True,
    label_duration_sec=None,  # transition mode: each cue switches state until the next cue.
    label_start_offset_sec=0.0,  # label switch starts exactly at cue onset.
    envelope_window_sec=0.025,
    onset_threshold=None,
    onset_min_interval_sec=0.50,
)

TRAIN = TrainingConfig(
    train_fraction=1.0,
    # hidden_size=256,
    hidden_size=128,
    num_layers=2,
    dropout=0.2,
    batch_size=64,
    epochs=20,
    lr=1e-3,
    seed=888,
)

OFFLINE_LABELED_DIR.mkdir(parents=True, exist_ok=True)
SWEEP_DIR.mkdir(parents=True, exist_ok=True)
print('Training run ID:', TRAIN_RUN_ID)
print('Test run ID:', TEST_RUN_ID)
print('Offline root:', OFFLINE_ROOT)
print('Offline run directory:', OFFLINE_RUN_DIR)
print('Training raw file:', TRAIN_RAW_NPZ)
print('Test raw file:', TEST_RAW_NPZ)
print('Offline labeled output:', OFFLINE_LABELED_DIR)
print('Sweep output:', SWEEP_DIR)
print('Preprocess tag:', PREPROCESS_TAG)
print('Software filters enabled:', APPLY_SOFTWARE_FILTERS)


## Preprocess raw files for this offline analysis


In [ ]:
TRAIN_LABELED_NPZ, train_cue_table = preprocess_recording(
    raw_npz=TRAIN_RAW_NPZ,
    output_npz=TRAIN_LABELED_NPZ,
    preprocess_config=PRE,
    label_config=LABELS,
)
TEST_LABELED_NPZ, test_cue_table = preprocess_recording(
    raw_npz=TEST_RAW_NPZ,
    output_npz=TEST_LABELED_NPZ,
    preprocess_config=PRE,
    label_config=LABELS,
)

preprocess_summary = labeled_preprocess_summary({
    'training': TRAIN_LABELED_NPZ,
    'test': TEST_LABELED_NPZ,
})
display(preprocess_summary)

print('Training cue table')
display(train_cue_table)
print('Test cue table')
display(test_cue_table)


## Inspect freshly labeled train/test data


In [ ]:
fig, axes = plot_labeled_recording(
    TRAIN_LABELED_NPZ,
    max_duration_sec=30.0,
    channel_names=EEG_CHANNEL_NAMES,
)
axes[0].set_title('Training labeled EEG preview')


In [ ]:
fig, axes = plot_labeled_recording(
    TEST_LABELED_NPZ,
    max_duration_sec=None,
    channel_names=EEG_CHANNEL_NAMES,
)
axes[0].set_title('Realtime trial labeled EEG preview')


## Run offline model sweep


In [ ]:
sweep_result = offline_train_test_sweep(
    train_labeled_npz=TRAIN_LABELED_NPZ,
    test_labeled_npz=TEST_LABELED_NPZ,
    output_dir=SWEEP_DIR,
    feature_modes=FEATURE_MODES,
    window_secs=WINDOW_SECS,
    stride_secs=STRIDE_SECS,
    training_config=TRAIN,
    label_modes=LABEL_MODES,
)

RANK_COLUMN = 'test_xcov_peak_coeff'
summary = rank_sweep_summary(sweep_result['summary'], rank_column=RANK_COLUMN)
print('Saved sweep summary:', sweep_result['summary_csv'])
print(f'Ranked variants by {RANK_COLUMN} descending; balanced accuracy is used only as a tie-breaker.')
display(summary)


## Compare variants


In [ ]:
TOP_N_VARIANTS = 5
plot_df = summary.head(TOP_N_VARIANTS).copy()
plot_df['variant_short'] = plot_df['variant'].str.replace('__', '\n', regex=False)
plot_df['test_xcov_peak_coeff'] = pd.to_numeric(plot_df['test_xcov_peak_coeff'], errors='coerce')
ax = plot_df.plot.bar(
    x='variant_short',
    y=['val_balanced_accuracy', 'test_balanced_accuracy', 'test_xcov_peak_coeff'],
    figsize=(35, 15),
    width=0.92,
)
ax.set_xlabel('Variant', fontsize=30, labelpad=16)
ax.set_ylabel('Accuracy / xcov', fontsize=30, labelpad=16)
ax.set_ylim(0.0, 1.05)
ax.set_title(
    f'Top {TOP_N_VARIANTS} offline sweep results for {RUN_ID} ranked by xcov peak coefficient',
    fontsize=38,
    pad=24,
)
ax.tick_params(axis='x', labelsize=27, rotation=90)
ax.tick_params(axis='y', labelsize=27)
for container in ax.containers:
    for bar in container:
        height = bar.get_height()
        if pd.notna(height):
            ax.text(
                bar.get_x() + bar.get_width() / 2.0,
                0.025,
                f'{float(height):.2f}',
                ha='center',
                va='bottom',
                fontsize=22,
                color='black',
                clip_on=True,
            )
ax.legend(fontsize=25, loc='center right')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()


## Inspect best variant predictions


In [ ]:
best = summary.iloc[0]
best_artifacts = load_test_variant_artifacts(best, TEST_LABELED_NPZ)
best_paths = best_artifacts['paths']
best_predictions = best_artifacts['predictions']
best_cue_delay_summary = best_artifacts['cue_delay_summary']
best_xcov_delay_summary = best_artifacts['xcov_delay_summary']
best_xcov_curve = best_artifacts['xcov_curve']

print('Best variant by xcov peak coefficient:', best['variant'])
print('Best xcov peak coefficient:', best.get('test_xcov_peak_coeff'))
print('Best xcov delay sec:', best.get('test_xcov_delay_sec'))
print('Best predictions:', best_paths['prediction_csv'])
print('Best aligned EEG/predictions:', best_paths['aligned_prediction_csv'])

display(best_predictions.tail())
if best_cue_delay_summary is not None:
    display(best_cue_delay_summary)
if best_xcov_delay_summary is not None:
    display(best_xcov_delay_summary)

fig, axes = plot_predictions_overlay(
    TEST_LABELED_NPZ,
    best_predictions,
    max_duration_sec=None,
    channel_names=EEG_CHANNEL_NAMES,
    show_true_labels=True,
    legend_loc='center right',
)
axes[0].set_title(f"Best offline variant by xcov peak coefficient on realtime trial: {best['variant']}")

if best_xcov_curve is not None and best_xcov_delay_summary is not None:
    fig, ax = plot_xcov_curve(
        best_xcov_curve,
        best_xcov_delay_summary,
        title=f"Prediction xcov lag for best xcov-ranked variant: {best['variant']}",
    )


## Compare variants by causal delay

The xcov lag is signed: positive delay means the prediction trace lags the label trace, while negative delay means the prediction trace leads the label trace. For a causal real-time model, negative delay should not be rewarded in the main delay ranking. This plot ranks variants with nonnegative xcov delay first, because those represent predictions that follow the label transition. Negative-delay variants remain in the full summary table but are placed after all nonnegative-delay variants here.


In [ ]:
DELAY_TOP_N_VARIANTS = 5
delay_plot_df = rank_sweep_by_causal_delay(summary).head(DELAY_TOP_N_VARIANTS).copy()
delay_plot_df['variant_short'] = delay_plot_df['variant'].str.replace('__', '\n', regex=False)

ax = delay_plot_df.plot.bar(
    x='variant_short',
    y=['val_balanced_accuracy', 'test_balanced_accuracy', 'test_xcov_peak_coeff'],
    figsize=(35, 15),
    width=0.92,
)
ax.set_xlabel('Variant', fontsize=30, labelpad=16)
ax.set_ylabel('Accuracy / xcov', fontsize=30, labelpad=16)
ax.set_ylim(0.0, 1.18)
ax.set_title(
    f'Top {DELAY_TOP_N_VARIANTS} offline sweep results for {RUN_ID} ranked by causal xcov delay',
    fontsize=38,
    pad=24,
)
ax.tick_params(axis='x', labelsize=27, rotation=90)
ax.tick_params(axis='y', labelsize=27)
for container in ax.containers:
    for bar in container:
        height = bar.get_height()
        if pd.notna(height):
            ax.text(
                bar.get_x() + bar.get_width() / 2.0,
                0.025,
                f'{float(height):.2f}',
                ha='center',
                va='bottom',
                fontsize=22,
                color='black',
                clip_on=True,
            )
ax.legend(fontsize=25, loc='center right')
ax.grid(True, axis='y', alpha=0.3)

for idx, value in enumerate(delay_plot_df['test_xcov_delay_sec']):
    if pd.notna(value):
        ax.text(
            idx,
            1.08,
            f'{float(value):+.2f}s',
            ha='center',
            va='bottom',
            fontsize=27,
            color='tab:red',
            fontweight='bold',
        )
plt.tight_layout()


## Inspect lowest causal-delay variant predictions

This repeats the same diagnostic printouts and plots for the fastest valid causal-delay combination. Negative xcov delays are excluded from this selection because a negative delay means the prediction trace leads the true label trace, which should not be treated as real-time prediction latency.


In [ ]:
lowest_delay = select_lowest_causal_delay_variant(summary)
lowest_delay_artifacts = load_test_variant_artifacts(lowest_delay, TEST_LABELED_NPZ)
lowest_delay_paths = lowest_delay_artifacts['paths']
lowest_delay_predictions = lowest_delay_artifacts['predictions']
lowest_delay_cue_delay_summary = lowest_delay_artifacts['cue_delay_summary']
lowest_delay_xcov_delay_summary = lowest_delay_artifacts['xcov_delay_summary']
lowest_delay_xcov_curve = lowest_delay_artifacts['xcov_curve']

print('Lowest causal-delay variant:', lowest_delay['variant'])
print('Lowest causal xcov delay sec:', lowest_delay.get('test_xcov_delay_sec'))
print('Lowest causal-delay xcov peak coefficient:', lowest_delay.get('test_xcov_peak_coeff'))
print('Lowest causal-delay predictions:', lowest_delay_paths['prediction_csv'])
print('Lowest causal-delay aligned EEG/predictions:', lowest_delay_paths['aligned_prediction_csv'])

display(lowest_delay_predictions.tail())
if lowest_delay_cue_delay_summary is not None:
    display(lowest_delay_cue_delay_summary)
if lowest_delay_xcov_delay_summary is not None:
    display(lowest_delay_xcov_delay_summary)

fig, axes = plot_predictions_overlay(
    TEST_LABELED_NPZ,
    lowest_delay_predictions,
    max_duration_sec=None,
    channel_names=EEG_CHANNEL_NAMES,
    show_true_labels=True,
    legend_loc='center right',
)
axes[0].set_title(
    f"Lowest causal-delay offline variant on realtime trial: {lowest_delay['variant']}"
)

if lowest_delay_xcov_curve is not None and lowest_delay_xcov_delay_summary is not None:
    fig, ax = plot_xcov_curve(
        lowest_delay_xcov_curve,
        lowest_delay_xcov_delay_summary,
        title=f"Prediction xcov lag for lowest causal-delay variant: {lowest_delay['variant']}",
    )


## Inspect any variant by window, stride, and label mode

Set the window, stride, and label mode below to pull that exact sweep result. This is useful for comparing the xcov-ranked or delay-ranked winners against another variant while keeping the same trace plot, xcov plot, and summary tables.


In [ ]:
COMPARE_FEATURE_MODE = 'filtered_signal'
COMPARE_WINDOW_SEC = 2.0
COMPARE_STRIDE_SEC = 0.05
COMPARE_LABEL_MODE = 'majority'  # 'endpoint' or 'majority'

compare_result = plot_offline_variant_trace_and_xcov(
    summary=summary,
    labeled_npz=TEST_LABELED_NPZ,
    feature_mode=COMPARE_FEATURE_MODE,
    window_sec=COMPARE_WINDOW_SEC,
    stride_sec=COMPARE_STRIDE_SEC,
    label_mode=COMPARE_LABEL_MODE,
    channel_names=EEG_CHANNEL_NAMES,
    max_duration_sec=None,
    show_true_labels=True,
    legend_loc='center right',
    title_prefix='Selected comparison variant',
)
compare_row = compare_result['row']
compare_artifacts = compare_result['artifacts']
compare_paths = compare_artifacts['paths']

print('Selected comparison variant:', compare_row['variant'])
print('Selected comparison xcov delay sec:', compare_row.get('test_xcov_delay_sec'))
print('Selected comparison xcov peak coefficient:', compare_row.get('test_xcov_peak_coeff'))
print('Selected comparison predictions:', compare_paths['prediction_csv'])
print('Selected comparison aligned EEG/predictions:', compare_paths['aligned_prediction_csv'])

display(compare_artifacts['predictions'].tail())
if compare_artifacts['cue_delay_summary'] is not None:
    display(compare_artifacts['cue_delay_summary'])
if compare_artifacts['xcov_delay_summary'] is not None:
    display(compare_artifacts['xcov_delay_summary'])
